### 🤖 Chatbot Evaluation Of RAG

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Model (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

How to evaluate the RAG applications using LangSmith:
1. How to create test datasets
2. How to run RAG application on those datasets
3. How to measure the application's performance using different evaluation metrics

- `Answer relevance`
- `Answer accuracy`
- `Retrieval quality`

1. Gathering Data Points
2. LLM as a Judge
3. Evaluation Metrics
4. Comparison Multiple LLM models

In [ ]:
# ------------------------------------------------------------
# 1. Load Environment Variables
# ------------------------------------------------------------

import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('groq_api_key')
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_api_key')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'

In [12]:
# from langsmith import Client

# client = Client()

# dataset_name = "Chatbot Evaluation"

# try:
#     dataset = client.read_dataset(dataset_name=dataset_name)
#     print("Using existing dataset:", dataset.name)

# except Exception:
#     dataset = client.create_dataset(dataset_name)
#     print("Created new dataset:", dataset.name)

Created new dataset: Chatbot Evaluation


In [15]:
# ------------------------------------------------------------
# 2. Create the Datapoints
# ------------------------------------------------------------

from langsmith import Client

client = Client()

# Define the dataset
dataset_name = 'Simple Chatbot Evaluation'
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id = dataset.id,
    examples = [
        {
            'inputs': {'question': 'What is LangChain?'},
            'outputs': {'answer': 'A framework for building LLM application'}
        },
        {
            'inputs': {'question': 'What is LangSmith?'},
            'outputs': {'answer': 'A platform for observing and evaluating LLM application'}
        },
        {
            'inputs': {'question': 'What is OpenAI?'},
            'outputs': {'answer': 'A company that creates Large Language Models'}
        },
        {
            'inputs': {'question': 'What is Google?'},
            'outputs': {'answer': 'A technology company known for search'}
        },
        {
            'inputs': {'question': 'What is Mistral?'},
            'outputs': {'answer': 'A company that creates Large Language Models'}
        }
    ]
)

{'example_ids': ['201a56b4-2ea6-4aae-8611-04d4771d155b',
  'fc093736-7451-49b5-b582-3758733b75f5',
  '12c61516-97d8-44af-bef2-3e9280240806',
  '03c13962-93f2-461a-87d4-f43ca981c1bc',
  '5b32344b-f4fc-4f56-8fa5-d3e9a7e3c28f'],
 'count': 5,
 'as_of': '2026-04-20T14:15:42.301060078Z'}

In [51]:
# ------------------------------------------------------------
# 3. Define Metrics (LLM as a Judge)
# ------------------------------------------------------------

import openai
from langsmith import wrappers

MODEL_NAME = 'llama-3.1-8b-instant'

openai_client = wrappers.wrap_openai(
    openai.OpenAI(
        api_key=os.getenv('groq_api_key'),
        base_url="https://api.groq.com/openai/v1"
    )
)

eval_instructions = 'You are an expert professor specialized in grading students answer to questions.'

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question: 
            {inputs['question']}
            Here is the real answer:
            {reference_outputs['answer']}
            You are grading the following predicted answer:
            {outputs['response']}
            Respond with CORRECT or INCORRECT:
            Grade:
        """

    response = openai_client.chat.completions.create(
        model=MODEL_NAME,  # or Groq model
        temperature=0,
        messages=[
            {'role': 'system', 'content': eval_instructions},
            {'role': 'user', 'content': user_content}
        ]
    ).choices[0].message.content
    
    return "CORRECT" in response.upper()

In [47]:
# ------------------------------------------------------------
# 4. Concisions - Checks whether the actual output is less than 2x the length of the expected result.
# ------------------------------------------------------------

def concision(outputs: dict, reference_outputs: dict):
    if 'response' not in outputs:
        return {"key": "concision", "score": 0}

    return {
        "key": "concision",
        "score": int(len(outputs['response']) < 2 * len(reference_outputs['answer']))
    }

In [52]:
# ------------------------------------------------------------
# 5. Run Evaluations
# ------------------------------------------------------------

default_instruction = "Response to the users question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str=MODEL_NAME):
    try:
        res = openai_client.chat.completions.create(
            model=model,
            temperature=0,
            messages=[
                {'role': 'system', 'content': default_instruction},
                {'role': 'user', 'content': question}
            ]
        )
        return res.choices[0].message.content
    except Exception as e:
        return f"ERROR: {str(e)}"

In [49]:
# Call my_app for every datapoints
def ls_target(inputs: dict) -> dict:
    return {'response': my_app(inputs['question'])}

In [53]:
# Run our evaluation
experiment_results = client.evaluate(
    ls_target, # Your AI system
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix=''
)

View the evaluation results for experiment: '-c1fe9899' at:
https://smith.langchain.com/o/a1921ab0-b36c-4cd2-97ec-4f0a042f060c/datasets/0c732993-0f66-4a83-9f09-5cb51ae5cb73/compare?selectedSessions=69e89744-4402-4da6-9c89-7ec75c6e6cd2




5it [00:04,  1.16it/s]
